# Pipeline de Análise Industrial - África e Médio Oriente

## Passos 1 a 9 com Geração Automática de Metadados

Este notebook executa o fluxo completo de análise de dados para 37 países da África e Médio Oriente, utilizando dados do Banco Mundial (WDI + WGI).

**Características:**
- Clonagem automática do repositório GitHub
- Extração automática via API (WDI + WGI)
- Agregação INNER JOIN + Dados Sintéticos (500 anos)
- Engenharia de Features avançada (lags, MA, deltas, interações)
- 7 Modelos (5 Clássicos + 2 Bayesianos)
- Interpretabilidade SHAP + Análise Geográfica
- **Sincronização simultânea de TODOS os ficheiros no Google Drive**

## 0. Configuração Inicial do Colab

In [ ]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive montado em /content/drive")

In [ ]:
# Instalar dependências
!pip install -q wbgapi pmdarima xgboost shap geopandas pymc arviz tensorflow scipy
print("✓ Dependências instaladas")

In [ ]:
# Clonar repositório GitHub
!git clone https://github.com/seu-usuario/seu-repositorio.git /content/repo
print("✓ Repositório clonado do GitHub")

In [ ]:
import os
import sys
import time
import shutil
from datetime import datetime
from pathlib import Path

# Configurar paths
REPO_DIR = '/content/repo'  # Repositório clonado do GitHub
PIPELINE_DIR = os.path.join(REPO_DIR, 'pipeline_africa_mo')  # Subdiretório do pipeline
DRIVE_DIR = '/content/drive/MyDrive/pipeline_africa_mo_resultados'  # Resultados no Drive

# Verificar se o pipeline está no subdiretório ou na raiz
if not os.path.exists(PIPELINE_DIR):
    PIPELINE_DIR = REPO_DIR  # Se estiver na raiz

os.chdir(PIPELINE_DIR)
sys.path.insert(0, PIPELINE_DIR)

# Criar diretório de resultados no Drive
os.makedirs(DRIVE_DIR, exist_ok=True)

print(f"✓ Diretório de trabalho: {PIPELINE_DIR}")
print(f"✓ Diretório de resultados (Drive): {DRIVE_DIR}")
print(f"\n  Ficheiros do pipeline:")
for f in sorted(os.listdir(PIPELINE_DIR)):
    if f.endswith('.py') or f.endswith('.md'):
        print(f"    {f}")

In [ ]:
# Função auxiliar para sincronizar TODOS os ficheiros gerados com o Drive
def sincronizar_todos_drive():
    """
    Sincroniza TODOS os diretórios de resultados com o Google Drive.
    Chamada após cada passo para backup simultâneo.
    """
    diretorio_saida = [
        'dados_brutos',
        'dados_limpos',
        'dados_agregados',
        'dados_sinteticos',
        'dados_engenharia',
        'eda_brutos',
        'eda_agregados',
        'modelos_treinados',
        'resultados_avaliacao',
        'analise_estrategias',
        'shap_analysis',
        'analise_geografica',
        'analise_avancada',
        'metadados'
    ]
    
    ficheiros_sincronizados = 0
    
    for dir_name in diretorio_saida:
        origem = os.path.join(PIPELINE_DIR, dir_name)
        if not os.path.exists(origem):
            continue
        
        destino = os.path.join(DRIVE_DIR, dir_name)
        os.makedirs(destino, exist_ok=True)
        
        # Sincronizar ficheiros
        for item in os.listdir(origem):
            src = os.path.join(origem, item)
            dst = os.path.join(destino, item)
            
            if os.path.isfile(src):
                shutil.copy2(src, dst)
                ficheiros_sincronizados += 1
            elif os.path.isdir(src):
                if os.path.exists(dst):
                    shutil.rmtree(dst)
                shutil.copytree(src, dst)
    
    return ficheiros_sincronizados

print("✓ Função de sincronização definida")

## 1. Extração de Dados (WDI + WGI)

In [ ]:
print("\n" + "="*70)
print("  PASSO 1: EXTRAÇÃO DE DADOS VIA API")
print("="*70)

t0 = time.time()
from passo1_extracao import executar_passo1
executar_passo1()

tempo_p1 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p1:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 2. EDA dos Dados Brutos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2: EDA DOS DADOS BRUTOS")
print("="*70)

t0 = time.time()
from passo2_eda_brutos import executar_passo2
executar_passo2()

tempo_p2 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 2.1. Limpeza de Dados

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.1: LIMPEZA DE DADOS")
print("="*70)

t0 = time.time()
from passo2_1_limpeza import executar_passo2_1
executar_passo2_1()

tempo_p2_1 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_1:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 2.2. Agregação INNER JOIN + Dados Sintéticos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.2: AGREGAÇÃO INNER JOIN + DADOS SINTÉTICOS (500 ANOS)")
print("="*70)

t0 = time.time()
from passo2_2_agregacao_sinteticos import executar_passo2_2
executar_passo2_2()

tempo_p2_2 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_2:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 2.3. EDA Agregados + Sintéticos

In [ ]:
print("\n" + "="*70)
print("  PASSO 2.3: EDA AGREGADOS + SINTÉTICOS")
print("="*70)

t0 = time.time()
from passo2_3_eda_agregados import executar_passo2_3
executar_passo2_3()

tempo_p2_3 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p2_3:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 3. Engenharia de Features

In [ ]:
print("\n" + "="*70)
print("  PASSO 3: ENGENHARIA DE FEATURES AVANÇADA")
print("="*70)

t0 = time.time()
from passo3_engenharia_features import executar_passo3
executar_passo3()

tempo_p3 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p3:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 4. Treinamento de 7 Modelos

In [ ]:
print("\n" + "="*70)
print("  PASSO 4: TREINAMENTO DE 7 MODELOS")
print("  (5 Clássicos: RF, XGBoost, TFT, SARIMAX, LSTM)")
print("  (2 Bayesianos: PartialPooling, CompletePooling)")
print("="*70)

t0 = time.time()
from passo4_treino_modelos import executar_passo4
executar_passo4()

tempo_p4 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p4:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 5. Avaliação de Performance

In [ ]:
print("\n" + "="*70)
print("  PASSO 5: AVALIAÇÃO DE PERFORMANCE")
print("="*70)

t0 = time.time()
from passo5_avaliacao import executar_passo5
executar_passo5()

tempo_p5 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p5:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 6. Análise de Estratégias

In [ ]:
print("\n" + "="*70)
print("  PASSO 6: ANÁLISE DE ESTRATÉGIAS")
print("="*70)

t0 = time.time()
from passo6_estrategias import executar_passo6
executar_passo6()

tempo_p6 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p6:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 7. Interpretabilidade (SHAP)

In [ ]:
print("\n" + "="*70)
print("  PASSO 7: INTERPRETABILIDADE (SHAP)")
print("="*70)

t0 = time.time()
from passo7_shap import executar_passo7
executar_passo7()

tempo_p7 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p7:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 8. Análise Geográfica

In [ ]:
print("\n" + "="*70)
print("  PASSO 8: ANÁLISE GEOGRÁFICA")
print("="*70)

t0 = time.time()
from passo8_geografica import executar_passo8
executar_passo8()

tempo_p8 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p8:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## 9. Análises Avançadas

In [ ]:
print("\n" + "="*70)
print("  PASSO 9: ANÁLISES AVANÇADAS")
print("="*70)

t0 = time.time()
from passo9_avancada import executar_passo9
executar_passo9()

tempo_p9 = time.time() - t0
print(f"\n  ⏱ Tempo: {tempo_p9:.1f}s")

# Sincronizar TODOS os ficheiros com Drive
n_sync = sincronizar_todos_drive()
print(f"  📁 {n_sync} ficheiros sincronizados com o Drive")

## Resumo Final

In [ ]:
print("\n" + "="*70)
print("  ✓ PIPELINE COMPLETO EXECUTADO COM SUCESSO!")
print("="*70)

# Resumo de tempos
tempos = {
    'Passo 1 (Extração)': tempo_p1,
    'Passo 2 (EDA Brutos)': tempo_p2,
    'Passo 2.1 (Limpeza)': tempo_p2_1,
    'Passo 2.2 (Agregação+Sintéticos)': tempo_p2_2,
    'Passo 2.3 (EDA Agregados)': tempo_p2_3,
    'Passo 3 (Features)': tempo_p3,
    'Passo 4 (Treino)': tempo_p4,
    'Passo 5 (Avaliação)': tempo_p5,
    'Passo 6 (Estratégias)': tempo_p6,
    'Passo 7 (SHAP)': tempo_p7,
    'Passo 8 (Geográfica)': tempo_p8,
    'Passo 9 (Avançada)': tempo_p9,
}

print("\n  Tempos de Execução:")
for passo, tempo in tempos.items():
    print(f"    {passo}: {tempo:.1f}s")

tempo_total = sum(tempos.values())
print(f"\n  ⏱ TEMPO TOTAL: {tempo_total:.1f}s ({tempo_total/60:.1f}m)")

print(f"\n  📁 Resultados:")
print(f"     Local (Colab): {PIPELINE_DIR}")
print(f"     Drive: {DRIVE_DIR}")

print(f"\n  📊 Ficheiros gerados:")
total_ficheiros = 0
for d in sorted(os.listdir(PIPELINE_DIR)):
    full = os.path.join(PIPELINE_DIR, d)
    if os.path.isdir(full) and not d.startswith('.'):
        n_files = len([f for f in os.listdir(full) if os.path.isfile(os.path.join(full, f))])
        if n_files > 0:
            print(f"     📁 {d}/  ({n_files} ficheiros)")
            total_ficheiros += n_files

print(f"\n  ✓ Total: {total_ficheiros} ficheiros gerados")
print(f"  ✓ Todos os ficheiros sincronizados com o Google Drive")

## Atualizar Repositório GitHub (Opcional)

In [ ]:
# OPCIONAL: Fazer push dos resultados para o GitHub
# Descomente as linhas abaixo se quiser atualizar o repositório

# !cd {REPO_DIR} && git add -A && git commit -m "Resultados pipeline $(date +%Y-%m-%d_%H:%M:%S)" && git push
# print("✓ Resultados enviados para o GitHub")